# CIFAR-10 Image Classification with CNN

A self-contained Convolutional Neural Network that classifies 32x32 color images into 10 classes
(airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

**How to use:** Google Colab → File → Upload notebook → Runtime → **Run all** (GPU runtime recommended).

Pipeline: Load CIFAR-10 → Normalize → One-hot labels → 70/15/15 split → CNN → Train → Evaluate.

In [ ]:
# Install extra packages (TensorFlow is pre-installed in Colab)
%pip install -q numpy matplotlib scikit-learn

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
# ---------- Configuration ----------
BATCH_SIZE   = 64
EPOCHS       = 30
LEARNING_RATE = 0.001
PATIENCE     = 5
VAL_SPLIT    = 0.15          # 70/15/15 train/val/test
IMAGE_SHAPE  = (32, 32, 3)
NUM_CLASSES  = 10

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

MODELS_DIR = "/content/models"
RESULTS_DIR = "/content/results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1) Data Pipeline

Load CIFAR-10, normalize pixels to [0, 1], one-hot encode labels, and split 70/15/15.

In [ ]:
def load_cifar10():
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    return x_train, y_train, x_test, y_test

def normalize(x):
    return x.astype("float32") / 255.0

def one_hot(y, num_classes=NUM_CLASSES):
    return keras.utils.to_categorical(y, num_classes)

def split_train_val(x, y, val_ratio=VAL_SPLIT, seed=SEED):
    # Split the official 50k train into 85% train / 15% validation (deterministic).
    # The official 10k test set is used untouched as the test split.
    indices = np.arange(len(x))
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    n_val = int(len(indices) * val_ratio)
    val_idx, train_idx = indices[:n_val], indices[n_val:]
    return x[train_idx], y[train_idx], x[val_idx], y[val_idx]

def prepare_data():
    x_train, y_train, x_test, y_test = load_cifar10()
    x_train, y_train, x_val, y_val = split_train_val(x_train, y_train)
    return {
        "x_train": normalize(x_train), "y_train": one_hot(y_train),
        "x_val":   normalize(x_val),   "y_val":   one_hot(y_val),
        "x_test":  normalize(x_test),  "y_test":  one_hot(y_test),
    }

data = prepare_data()
for name, arr in data.items():
    print(f"{name}: {arr.shape}")

## 2) Visualize Sample Images

In [ ]:
def show_sample_grid(x, y, class_names=CLASS_NAMES, num_samples=15):
    cols = 5
    rows = int(np.ceil(num_samples / cols)) or 1
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2.4))
    axes = np.atleast_1d(axes).ravel()
    y_labels = np.argmax(y, axis=-1) if y.ndim == 2 else y
    for i in range(num_samples):
        axes[i].imshow(x[i])
        axes[i].set_title(class_names[y_labels[i]], fontsize=8)
        axes[i].axis("off")
    for ax in axes[num_samples:]:
        ax.axis("off")
    plt.suptitle("Sample Training Images", y=1.02)
    plt.tight_layout()
    plt.show()

show_sample_grid(data["x_train"], data["y_train"])

## 3) Build the CNN Model

```
Conv(32) → BN → Conv(32) → MaxPool → Dropout
Conv(64) → BN → Conv(64) → MaxPool → Dropout
Conv(128) → BN → Conv(128) → MaxPool → Dropout
Flatten → Dense(256) → BN → Dropout(0.5) → Dense(10, softmax)
```

In [ ]:
def build_cnn(input_shape=IMAGE_SHAPE, num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cifar10_cnn")
    return model

model = build_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 4) Train the Model

- **Augmentation**: rotation, translation, zoom, horizontal flip — applied on the fly
- **EarlyStopping**: patience=5, restores best weights
- **ModelCheckpoint**: keeps best `val_accuracy` model
- **ReduceLROnPlateau**: halves LR when validation loss stagnates

In [ ]:
augmenter = keras.Sequential([
    layers.RandomRotation(0.15, fill_mode="nearest"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="nearest"),
    layers.RandomZoom(0.1, 0.1, fill_mode="nearest"),
    layers.RandomFlip("horizontal"),
], name="augmentation")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "img_class_best.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    augmenter(data["x_train"]),
    data["y_train"],
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(data["x_val"], data["y_val"]),
    callbacks=callbacks_list,
    verbose=1,
)

In [ ]:
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="train")
    ax1.plot(history.history["val_loss"], label="val")
    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history.history["accuracy"], label="train")
    ax2.plot(history.history["val_accuracy"], label="val")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "training_history.png"), dpi=120, bbox_inches="tight")
    plt.show()

plot_training_history(history)

## 5) Evaluate the Model

Test accuracy, confusion matrix, classification report, and misclassified samples.

In [ ]:
test_loss, test_acc = model.evaluate(data["x_test"], data["y_test"], verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_true = np.argmax(data["y_test"], axis=-1)
y_pred = np.argmax(model.predict(data["x_test"], verbose=0), axis=-1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, "%d" % cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {acc:.4f}\n")
print(report)

with open(os.path.join(RESULTS_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(f"Test Accuracy: {acc:.4f}\n\n")
    f.write(report)

mis_idx = np.where(y_true != y_pred)[0]
idx = mis_idx[:10]
cols, rows = 5, 2
fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2.4))
axes = np.atleast_1d(axes).ravel()
for ax_i, i in enumerate(idx):
    axes[ax_i].imshow(data["x_test"][i])
    axes[ax_i].set_title(f"T:{CLASS_NAMES[y_true[i]]}\nP:{CLASS_NAMES[y_pred[i]]}",
                         fontsize=8, color="red")
    axes[ax_i].axis("off")
for ax_i in range(len(idx), len(axes)):
    axes[ax_i].axis("off")
plt.suptitle("Misclassified Samples", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "misclassified.png"), dpi=120, bbox_inches="tight")
plt.show()

## 6) Save the Model

The best checkpoint was already saved by `ModelCheckpoint`. Saved artifacts:

In [ ]:
model.save(os.path.join(MODELS_DIR, "cifar10_cnn_final.keras"))
print("Model saved to:", os.path.join(MODELS_DIR, "cifar10_cnn_final.keras"))
print("\nSaved artifacts:")
for root, _, files in os.walk(MODELS_DIR):
    for f in files:
        print(os.path.join(root, f))
for root, _, files in os.walk(RESULTS_DIR):
    for f in files:
        print(os.path.join(root, f))

## 7) (Optional) Save Everything to Google Drive

Run the cell below to copy `models/` and `results/` to your Drive so they persist after the session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/CIFAR10_Project"
!cp -r models results "/content/drive/MyDrive/CIFAR10_Project/"
print("Saved to /content/drive/MyDrive/CIFAR10_Project/")